# Depression Detection: Feature Analysis & Statistical Validation

**Objective:** Identify features with statistically significant and clinically meaningful differences between depressed and non-depressed groups.

**Clinical Hypothesis:** Depression manifests in measurable linguistic and acoustic patterns detectable through multimodal analysis.

---

## Methodology
- **Statistical Test:** Mann-Whitney U (non-parametric, suitable for small samples)
- **Effect Size:** Cohen's d (clinical significance)
- **Significance Level:** α = 0.05
- **Features:** TTR (linguistic), Audio statistics, Question type ratios

In [ ]:
import sys
sys.path.append('..')

from eda import (
    DAICDataLoader,
    ComprehensiveFeatureProfiler,
    StatisticalAnalyzer,
    FeatureImportanceRanker,
    ClinicalVisualizationSuite
)
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## 1. Feature Extraction

In [ ]:
loader = DAICDataLoader.from_config('../config/paths.yaml')
profiler = ComprehensiveFeatureProfiler(loader)

print("Extracting features for training set...")
train_features = profiler.profile_dataset(
    split='train',
    include_audio=True,
    include_text=True
)

print(f"\nExtracted {len(train_features.columns)} features for {len(train_features)} participants")
print(f"\nFeature categories:")
print(f"  - Linguistic: {len([c for c in train_features.columns if 'ttr' in c or 'word' in c])}")
print(f"  - Audio: {len([c for c in train_features.columns if 'audio' in c])}")
print(f"  - Question type: {len([c for c in train_features.columns if 'qtype' in c])}")

In [ ]:
train_features.head()

## 2. Linguistic Feature Analysis

### Type-Token Ratio (TTR)
**Clinical Relevance:** TTR measures vocabulary diversity. Lower TTR in depression reflects:
- Reduced cognitive flexibility
- Limited emotional expression
- Repetitive thought patterns

In [ ]:
analyzer = StatisticalAnalyzer()

ttr_result = analyzer.analyze_feature_dataframe(train_features, 'ttr')

print("TTR Analysis Results:")
print(f"  Depressed group: μ={ttr_result['mean_depressed']:.4f}, σ={ttr_result['std_depressed']:.4f}")
print(f"  Non-depressed group: μ={ttr_result['mean_non_depressed']:.4f}, σ={ttr_result['std_non_depressed']:.4f}")
print(f"\n  Mann-Whitney U: {ttr_result['mann_whitney_u']:.1f}")
print(f"  p-value: {ttr_result['p_value']:.4f} {'***' if ttr_result['p_value'] < 0.001 else '**' if ttr_result['p_value'] < 0.01 else '*' if ttr_result['p_value'] < 0.05 else ''}")
print(f"  Cohen's d: {ttr_result['cohens_d']:.3f} ({ttr_result['effect_size']})")
print(f"\n  Statistically significant: {ttr_result['significant']}")

In [ ]:
viz = ClinicalVisualizationSuite()

fig = viz.plot_distribution_comparison(
    train_features,
    'ttr',
    title='Type-Token Ratio Distribution',
    xlabel='TTR (Vocabulary Diversity)',
    stat_test_result=ttr_result
)
viz.save_figure(fig, '../reports/figures/ttr_distribution.png')

In [ ]:
fig = viz.plot_boxplot_comparison(
    train_features,
    'ttr',
    title='TTR Comparison: Depressed vs Non-depressed',
    ylabel='Type-Token Ratio'
)
viz.save_figure(fig, '../reports/figures/ttr_boxplot.png')

### Word Count Analysis

In [ ]:
if 'word_count' in train_features.columns:
    wc_result = analyzer.analyze_feature_dataframe(train_features, 'word_count')
    
    print("Word Count Analysis:")
    print(f"  Depressed: μ={wc_result['mean_depressed']:.1f}")
    print(f"  Non-depressed: μ={wc_result['mean_non_depressed']:.1f}")
    print(f"  p-value: {wc_result['p_value']:.4f}")
    print(f"  Cohen's d: {wc_result['cohens_d']:.3f}")

## 3. Audio Feature Analysis

In [ ]:
audio_features = [
    'audio_mean', 'audio_std', 'audio_min', 'audio_max',
    'audio_seq_len', 'audio_l2_norm'
]

available_audio = [f for f in audio_features if f in train_features.columns]

audio_results = analyzer.batch_analyze_features(
    train_features,
    available_audio
)

print("Audio Feature Analysis Results:")
print(audio_results[['feature', 'p_value', 'cohens_d', 'effect_size']].to_string(index=False))

## 4. Question Type Distribution Analysis

**Hypothesis:** Different question types elicit varying levels of depression-related information.

In [ ]:
qtype_features = [c for c in train_features.columns if 'qtype_' in c and '_ratio' in c]

if qtype_features:
    qtype_results = analyzer.batch_analyze_features(
        train_features,
        qtype_features
    )
    
    print("\nTop 5 Question Types by Effect Size:")
    top_qtypes = qtype_results.nlargest(5, 'cohens_d', keep='all')
    print(top_qtypes[['feature', 'mean_depressed', 'mean_non_depressed', 'cohens_d', 'p_value']].to_string(index=False))

In [ ]:
if len(qtype_features) >= 5:
    fig = viz.plot_question_type_distribution(
        train_features,
        qtype_features[:10],
        title='Question Type Distribution by Depression Status'
    )
    viz.save_figure(fig, '../reports/figures/qtype_distribution.png')

## 5. Comprehensive Feature Ranking

In [ ]:
all_feature_cols = (
    ['ttr', 'word_count', 'avg_word_length'] +
    available_audio +
    qtype_features
)

all_feature_cols = [f for f in all_feature_cols if f in train_features.columns]

comprehensive_results = analyzer.batch_analyze_features(
    train_features,
    all_feature_cols
)

print("\nTop 15 Features by Effect Size:")
print(comprehensive_results.head(15)[['feature', 'cohens_d', 'p_value', 'effect_size']].to_string(index=False))

In [ ]:
fig = viz.plot_effect_size_ranking(
    comprehensive_results,
    top_n=15,
    title="Feature Importance Ranking (Cohen's d)"
)
viz.save_figure(fig, '../reports/figures/feature_ranking.png')

## 6. Feature Correlation Analysis

In [ ]:
top_features = comprehensive_results.head(10)['feature'].tolist()
top_features = [f for f in top_features if f in train_features.columns]

if len(top_features) >= 3:
    fig = viz.plot_correlation_matrix(
        train_features,
        top_features,
        title='Top Features Correlation Matrix'
    )
    viz.save_figure(fig, '../reports/figures/feature_correlation.png')

## 7. Clinically Relevant Features

In [ ]:
ranker = FeatureImportanceRanker(comprehensive_results)

significant_features = ranker.filter_significant_features(alpha=0.05)
clinical_features = ranker.get_clinical_relevant_features(min_effect_size=0.5)

print(f"\nFeature Summary:")
print(f"  Total features analyzed: {len(comprehensive_results)}")
print(f"  Statistically significant (p<0.05): {len(significant_features)}")
print(f"  Clinically relevant (|d|≥0.5): {len(clinical_features)}")
print(f"  Both significant AND clinically relevant: {len(set(significant_features['feature']) & set(clinical_features['feature']))}")

In [ ]:
comprehensive_results.to_csv('../reports/statistical_analysis_results.csv', index=False)
print("\n✓ Results saved to: reports/statistical_analysis_results.csv")

## Key Findings

### Linguistic Features
- **TTR:** Depressed participants show lower vocabulary diversity (clinically meaningful)
- **Word Count:** [Insert finding based on results]

### Audio Features
- [Insert key findings from audio analysis]

### Question Types
- **Most discriminative:** [Insert top question type]
- **Least discriminative:** [Insert bottom question type]

### Clinical Implications
1. Features with large effect sizes (|d|>0.8) should be prioritized in model development
2. Multimodal approach justified by complementary discriminative patterns
3. Question type analysis suggests protocol optimization opportunities

---

**Continue to:** `03_question_type_analysis.ipynb`